# ETL Gold - training_dataset_v0

Construye una tabla diaria lista para entrenamiento de modelos ML para `codigoestacao = 74100000`.

In [ ]:
from datetime import date, timedelta

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

LEVEL_TABLE = 'weather.silver.river_levels_daily'
TEMP_TABLE = 'weather.silver.temperature_daily'
RAIN_TABLE = 'weather.silver.rainfall_daily'
DISCHARGE_TABLE = 'weather.silver.river_discharge_daily'
SUBCUENCA_TABLE = 'weather.silver.estacion_subcuenca'
PRECIP_GRID_TABLE = 'weather.silver.precip_grid_daily'  # MERGE (CPTEC), Decision 033
TEMP_GRID_TABLE = 'weather.silver.temp_grid_daily'  # SAMeT (CPTEC), Decision 033
QUALITY_TABLE = 'weather.silver.attribute_quality'
FORECAST_CF_TABLE = 'weather.silver.ecmwf_forecast_cf_subcuenca'  # ECMWF determinista (Decision 048)
FORECAST_PF_TABLE = 'weather.silver.ecmwf_forecast_pf_subcuenca'  # ECMWF ensemble, 50 miembros
TARGET_TABLE = 'weather.gold.training_dataset_v0'
TARGET_STATION = '74100000'
PUNTO_PREDICCION = 'ana_74100000'
SUBCUENCAS = ['alta_frontera', 'intermedia_paso_libres', 'baja_salto_grande']
DATASET_FLOOR = date(2000, 1, 1)  # R1 (Decision 019, enmienda): Gold arranca en 2000-01-01
FORECAST_SUBCUENCA = 'alta_frontera'  # a Gold solo pasa la cuenca alta de Brasil (Decision 018)
FORECAST_LEAD_DAYS = list(range(1, 16))  # pasos de 24h a 360h del pronostico TIGGE

try:
    dbutils.widgets.dropdown('load_mode', 'incremental', ['full', 'incremental'])
    dbutils.widgets.text('incremental_lookback_days', '14')
    load_mode = dbutils.widgets.get('load_mode')
    incremental_lookback_days = int(dbutils.widgets.get('incremental_lookback_days'))
except Exception:
    load_mode = 'incremental'
    incremental_lookback_days = 14

print(f'load_mode={load_mode}, incremental_lookback_days={incremental_lookback_days}')

In [ ]:
def quality_is_usable(source_table, attribute_name):
    rows = (
        spark.table(QUALITY_TABLE)
        .filter(F.col('source_table') == F.lit(source_table))
        .filter(F.col('attribute_name') == F.lit(attribute_name))
        .filter(F.col('grain') == F.lit('global_source_daily'))
        .orderBy(F.col('evaluated_at').desc_nulls_last())
        .limit(1)
        .collect()
    )
    if not rows:
        return False
    return bool(rows[0]['is_usable'])


def build_calendar(levels_df):
    bounds = levels_df.agg(F.min('fecha').alias('min_fecha'), F.max('fecha').alias('max_fecha')).first()
    if bounds['min_fecha'] is None or bounds['max_fecha'] is None:
        raise ValueError('No level data available for Gold calendar')

    calendar_start = max(bounds['min_fecha'], DATASET_FLOOR)
    if calendar_start > bounds['max_fecha']:
        raise ValueError(f'No level data on/after DATASET_FLOOR={DATASET_FLOOR}')

    return spark.sql(
        f"SELECT explode(sequence(to_date('{calendar_start}'), to_date('{bounds['max_fecha']}'), interval 1 day)) AS fecha"
    )


def target_window(dataset_df):
    source_bounds = dataset_df.agg(F.min('fecha').alias('min_fecha'), F.max('fecha').alias('max_fecha')).first()
    if source_bounds['min_fecha'] is None or source_bounds['max_fecha'] is None:
        raise ValueError('No Gold rows generated')

    if load_mode == 'full':
        return source_bounds['min_fecha'], source_bounds['max_fecha']

    max_target_fecha = (
        spark.table(TARGET_TABLE)
        .filter(F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION))
        .agg(F.max('fecha').alias('max_fecha'))
        .first()['max_fecha']
    )
    if max_target_fecha is None:
        return source_bounds['min_fecha'], source_bounds['max_fecha']

    changed_min = max(source_bounds['min_fecha'], max_target_fecha - timedelta(days=incremental_lookback_days))
    changed_max = source_bounds['max_fecha']
    window_start = max(source_bounds['min_fecha'], changed_min - timedelta(days=14))
    window_end = min(source_bounds['max_fecha'], changed_max + timedelta(days=7))
    return window_start, window_end


def replace_target_window(dataset_df, window_start, window_end):
    target_delta = DeltaTable.forName(spark, TARGET_TABLE)
    if load_mode == 'full':
        delete_condition = f"punto_prediccion = '{PUNTO_PREDICCION}'"
    else:
        delete_condition = f"punto_prediccion = '{PUNTO_PREDICCION}' AND fecha >= DATE '{window_start}' AND fecha <= DATE '{window_end}'"

    print(f'Deleting Gold rows with condition: {delete_condition}')
    target_delta.delete(delete_condition)

    rows_to_write = dataset_df.filter((F.col('fecha') >= F.lit(window_start)) & (F.col('fecha') <= F.lit(window_end)))
    count_rows = rows_to_write.count()
    print(f'Appending {count_rows} Gold rows')
    if count_rows > 0:
        # mergeSchema: incorporar una familia nueva de features (las columnas de pronostico
        # ECMWF) no puede exigir un ALTER TABLE manual antes del primer run del job.
        rows_to_write.write.format('delta').option('mergeSchema', 'true').mode('append').saveAsTable(TARGET_TABLE)

In [ ]:
levels = (
    # Se sigue leyendo el nivel porque define el calendario (la espina de fechas del
    # dataset), pero sus columnas NO se publican: Decision 040.
    spark.table(LEVEL_TABLE)
    .filter(F.col('codigoestacao') == F.lit(TARGET_STATION))
    .select(
        'fecha',
        'codigoestacao',
    )
)

calendar = build_calendar(levels)

# Sub-cuenca alta_frontera (weather.silver.estacion_subcuenca): universo compartido por
# lluvia, caudal y temperatura -- una estacion cae en alta_frontera independientemente de
# que fuente la reporte.
alta_frontera_universe = (
    spark.table(SUBCUENCA_TABLE)
    .filter(F.col('subcuenca') == F.lit('alta_frontera'))
    .select('codigoestacao')
    .distinct()
)
alta_frontera_station_count = alta_frontera_universe.count()

# Temperatura agregada por sub-cuenca (R8, Decision 019; Decision 025): antes `temp_global`
# promediaba TODOS los aeropuertos METAR sin ningun join contra estacion_subcuenca -- violaba
# R2 igual que el bug de lluvia que corrigieron las Decisiones 023/024, solo que sin datos
# faltantes de por medio (el numero resultante era temperatura nacional, no de la cuenca).
# weather.silver.temperature_daily.estacion_id se junta contra el mismo universo de
# alta_frontera que usa lluvia; en la practica solo estaciones INMET caen ahi (los 4
# aeropuertos METAR estan geograficamente fuera de las tres sub-cuencas), asi que no hace
# falta ninguna regla de prioridad entre fuentes -- no compiten por la misma sub-cuenca.
temp_alta_frontera = (
    spark.table(TEMP_TABLE).alias('t')
    .join(alta_frontera_universe.alias('sc'), F.col('t.estacion_id') == F.col('sc.codigoestacao'), 'inner')
    .groupBy('fecha')
    .agg(
        F.avg('temp_media_c').alias('temp_media_c'),
        F.min('temp_min_c').alias('temp_min_c'),
        F.max('temp_max_c').alias('temp_max_c'),
        F.countDistinct(F.when(F.col('temp_media_c').isNotNull(), F.col('estacion_id'))).cast('bigint').alias('temp_agregado_alta_frontera_station_count'),
    )
    .withColumn(
        'temp_agregado_alta_frontera_cobertura_pct',
        F.when(F.lit(alta_frontera_station_count) > 0, F.col('temp_agregado_alta_frontera_station_count') / F.lit(alta_frontera_station_count)),
    )
)

# Lluvia agregada por sub-cuenca (R8, Decision 019): sin umbral de exclusion, se publica
# toda estacion con dato real (ver ETL_Silver_Rainfall_Daily.ipynb). Mismo join que el
# agregado de caudal (weather.silver.estacion_subcuenca). lluvia_acumulada_mm mantiene su
# nombre historico pero corrige su alcance: antes sumaba las ~392 estaciones de toda la cuenca
# (violaba R2), ahora solo las de alta_frontera. La cobertura real viaja como columna
# (lluvia_agregado_alta_frontera_station_count/_cobertura_pct) en vez de un porton
# binario global.
rain_alta_frontera = (
    spark.table(RAIN_TABLE).alias('r')
    .join(alta_frontera_universe.alias('sc'), 'codigoestacao', 'inner')
    .groupBy('fecha')
    .agg(
        F.sum('lluvia_acumulada_mm').alias('lluvia_acumulada_mm'),
        F.countDistinct(F.when(F.col('lluvia_acumulada_mm').isNotNull(), F.col('codigoestacao'))).cast('bigint').alias('lluvia_agregado_alta_frontera_station_count'),
    )
    .withColumn(
        'lluvia_agregado_alta_frontera_cobertura_pct',
        F.when(F.lit(alta_frontera_station_count) > 0, F.col('lluvia_agregado_alta_frontera_station_count') / F.lit(alta_frontera_station_count)),
    )
)

# Observacion en grilla de CPTEC/INPE (Decision 033): MERGE (lluvia) y SAMeT (temperatura),
# media areal de alta_frontera calculada en Silver (ETL_Silver_CPTEC_Grid_Daily.ipynb) con
# weather.silver.grid_subcuenca. Conviven con los agregados por estacion de arriba: son otra
# medicion de la misma variable, con cobertura espacial completa de la sub-cuenca y ~1 dia de
# latencia. La ventana diaria de MERGE es 12Z(D-1)->12Z(D) (no dia calendario); la de SAMeT es el
# dia calendario UTC, igual que temperature_daily. `*_es_preliminar` avisa que la fila todavia
# viene de la version preliminar del archivo de origen (se regenera al mes siguiente / a los 7 dias).
merge_alta_frontera = (
    spark.table(PRECIP_GRID_TABLE)
    .filter((F.col('subcuenca') == F.lit('alta_frontera')) & (F.col('fuente') == F.lit('merge')))
    .select(
        'fecha',
        F.col('prec_media_mm').alias('lluvia_merge_alta_frontera_mm'),
        F.col('prec_max_mm').alias('lluvia_merge_alta_frontera_max_mm'),
        F.col('pluviometros').alias('lluvia_merge_alta_frontera_pluviometros'),
        F.col('cobertura_pct').alias('lluvia_merge_alta_frontera_cobertura_pct'),
        F.col('es_preliminar').alias('lluvia_merge_alta_frontera_es_preliminar'),
    )
)

samet_alta_frontera = (
    spark.table(TEMP_GRID_TABLE)
    .filter((F.col('subcuenca') == F.lit('alta_frontera')) & (F.col('fuente') == F.lit('samet')))
    .select(
        'fecha',
        F.col('temp_media_c').alias('temp_samet_alta_frontera_media_c'),
        F.col('temp_max_c').alias('temp_samet_alta_frontera_max_c'),
        F.col('temp_min_c').alias('temp_samet_alta_frontera_min_c'),
        F.col('cobertura_pct').alias('temp_samet_alta_frontera_cobertura_pct'),
        F.col('es_preliminar').alias('temp_samet_alta_frontera_es_preliminar'),
    )
)

# Caudal de la estacion target (Decision D2: caudal es el target principal, ver
# docs/decisions.md (Decision 017)). El nivel se mantiene intacto arriba. curva_vigencia_extendida
# (R4, Decision 019 enmienda) viaja tal cual desde Silver.
discharge_usable = quality_is_usable(DISCHARGE_TABLE, 'caudal_m3s')
discharge_target = (
    spark.table(DISCHARGE_TABLE)
    .filter(F.col('codigoestacao') == F.lit(TARGET_STATION))
    .select(
        'fecha',
        F.col('caudal_m3s').alias('caudal_actual_m3s'),
        'caudal_metodo', 'caudal_extrapolado', 'distancia_fuera_rango_cm', 'supera_aforo_maximo', 'caudal_confiable',
        'curva_vigencia_extendida',
    )
)

# Agregados de caudal por sub-cuenca (SIG/subcuencas_modelo.geojson via weather.silver.estacion_subcuenca):
# el caudal es fisicamente aditivo entre estaciones (el nivel no), asi que sumar el caudal
# de las estaciones de una sub-cuenca da el aporte total aguas arriba de ese punto. Este JOIN
# siempre fue dinamico contra estacion_subcuenca, no hardcodeado al grupo A: desde la Decision 024
# (resiembra completa del inventario ANA en estacion_subcuenca) tambien contribuyen 14 de las 40
# estaciones "grupo B" (con curva, fuera de la cuenca alta segun el barrido de la Fase 2) que la
# union espacial real ubica dentro de alta_frontera -- verificado y documentado en la Decision 028
# (Fase 7 del roadmap). Las columnas de intermedia_paso_libres/baja_salto_grande tambien dejaron
# de estar en NULL por el mismo motivo (23 y 2 estaciones del grupo B respectivamente), aunque
# esas dos sub-cuencas siguen fuera del alcance de la tesis (Decision 018).
subcuenca_daily = (
    spark.table(DISCHARGE_TABLE).alias('d')
    .join(spark.table(SUBCUENCA_TABLE).alias('sc'), 'codigoestacao', 'inner')
    .groupBy('fecha', 'subcuenca')
    .agg(
        F.sum('caudal_m3s').alias('caudal_agregado_m3s'),
        F.avg(F.col('caudal_confiable').cast('double')).alias('confiable_pct'),
    )
)

subcuenca_wide = calendar.select('fecha')
for subcuenca in SUBCUENCAS:
    one = (
        subcuenca_daily.filter(F.col('subcuenca') == F.lit(subcuenca))
        .select(
            'fecha',
            F.col('caudal_agregado_m3s').alias(f'caudal_agregado_{subcuenca}_m3s'),
            F.col('confiable_pct').alias(f'caudal_agregado_{subcuenca}_confiable_pct'),
        )
    )
    subcuenca_wide = subcuenca_wide.join(one, 'fecha', 'left')

# Pronostico ECMWF agregado a la sub-cuenca (Decision 048). Solo entra `alta_frontera`: es el
# alcance de la tesis (Decision 018) y el mismo universo que usan todas las demas columnas de
# esta tabla. Silver conserva las tres sub-cuencas y los 50 miembros; el recorte y el colapso a
# un numero pasan aca, que es donde se pueden cambiar sin recalcular nada aguas arriba.
#
# `tp_mm_medio` viene ACUMULADO desde el inicio del pronostico (asi lo entrega TIGGE), asi que la
# lluvia pronosticada para el dia de adelanto d es la diferencia entre el paso 24*d y el 24*(d-1).
# Publicar el acumulado crudo daria 15 columnas fuertemente colineales y ninguna en la unidad
# "mm que caen ese dia", que es la que necesita un modelo hidrologico.
#
# La metrica sobre los miembros es la MEDIA, y es provisional: promediar el ensemble tira
# justamente la dispersion que lo hace valioso. El reemplazo natural (P90, maximo, fraccion de
# miembros sobre umbral) se implementa cambiando el F.avg de abajo -- Silver ya guarda todo lo
# necesario.
def forecast_lead_day_features(table, prefix, con_miembros):
    if not spark.catalog.tableExists(table):
        print(f'{table} no existe todavia; {prefix} se publica en NULL')
        return None

    src = spark.table(table).filter(F.col('subcuenca_nombre') == F.lit(FORECAST_SUBCUENCA))

    w_step = Window.partitionBy('run_date', 'run_time', 'number').orderBy('step_hours')
    incrementos = (
        src
        .withColumn('tp_acum_prev', F.lag('tp_mm_medio').over(w_step))
        .filter(F.col('step_hours') > 0)
        .withColumn('tp_dia_mm', F.col('tp_mm_medio') - F.coalesce(F.col('tp_acum_prev'), F.lit(0.0)))
        .withColumn('lead_day', (F.col('step_hours') / F.lit(24)).cast('int'))
        .filter(F.col('lead_day').isin(FORECAST_LEAD_DAYS))
    )

    lead_wide = (
        incrementos
        .groupBy('run_date', 'lead_day')
        .agg(F.avg('tp_dia_mm').alias('tp_dia_mm'))
        .groupBy('run_date')
        .pivot('lead_day', FORECAST_LEAD_DAYS)
        .agg(F.first('tp_dia_mm'))
        .select(
            F.col('run_date').alias('fecha'),
            *[F.col(str(d)).alias(f'{prefix}_tp_mm_d{d}') for d in FORECAST_LEAD_DAYS],
        )
    )

    # Los acumulados a 3 y 7 dias se leen directo del valor acumulado de los pasos 72h y 168h,
    # en vez de volver a sumar los incrementos: menos aritmetica y sin error de redondeo.
    acum_wide = (
        src.filter(F.col('step_hours').isin([72, 168]))
        .groupBy('run_date', 'step_hours')
        .agg(F.avg('tp_mm_medio').alias('tp_acum_mm'))
        .groupBy('run_date')
        .pivot('step_hours', [72, 168])
        .agg(F.first('tp_acum_mm'))
        .select(
            F.col('run_date').alias('fecha'),
            F.col('72').alias(f'{prefix}_tp_acum_3d_mm'),
            F.col('168').alias(f'{prefix}_tp_acum_7d_mm'),
        )
    )

    out = lead_wide.join(acum_wide, 'fecha', 'left')
    if con_miembros:
        # Cuantos miembros respaldan la media de ese dia: un dia con 12 de 50 miembros no es
        # comparable con uno completo, y sin esta columna el modelo no puede distinguirlos.
        cobertura = (
            src.groupBy('run_date')
            .agg(F.countDistinct('number').alias(f'{prefix}_n_miembros'))
            .select(F.col('run_date').alias('fecha'), f'{prefix}_n_miembros')
        )
        out = out.join(cobertura, 'fecha', 'left')
    return out


FORECAST_MODELOS = [
    (FORECAST_CF_TABLE, 'ecmwf_cf', False),
    (FORECAST_PF_TABLE, 'ecmwf_pf', True),
]

forecast_columns = []
forecast_frames = []
for _tabla, _prefix, _con_miembros in FORECAST_MODELOS:
    _cols = (
        [f'{_prefix}_tp_mm_d{d}' for d in FORECAST_LEAD_DAYS]
        + [f'{_prefix}_tp_acum_3d_mm', f'{_prefix}_tp_acum_7d_mm']
        + ([f'{_prefix}_n_miembros'] if _con_miembros else [])
    )
    forecast_columns += _cols
    forecast_frames.append((forecast_lead_day_features(_tabla, _prefix, _con_miembros), _cols))

base = (
    calendar.join(levels, 'fecha', 'left')
    .join(temp_alta_frontera, 'fecha', 'left')
    .join(rain_alta_frontera, 'fecha', 'left')
    .join(discharge_target, 'fecha', 'left')
    .join(merge_alta_frontera, 'fecha', 'left')
    .join(samet_alta_frontera, 'fecha', 'left')
    .join(subcuenca_wide, 'fecha', 'left')
    .withColumn('punto_prediccion', F.lit(PUNTO_PREDICCION))
    .withColumn('codigoestacao', F.lit(TARGET_STATION))
    .withColumn('lluvia_is_usable', F.lit(None).cast('boolean'))  # deprecado (R8): cobertura real en lluvia_agregado_alta_frontera_cobertura_pct
    .withColumn('caudal_registros_validos', F.when(F.col('caudal_actual_m3s').isNotNull(), F.lit(1)).otherwise(F.lit(0)).cast('bigint'))
)

# El pronostico se une por fecha = run_date: la fila de un dia lleva el pronostico EMITIDO ese
# dia, que es exactamente la informacion disponible en ese momento para predecir hacia adelante.
# Antes de 2006-10 (primer dia de TIGGE) las columnas quedan en NULL por construccion.
for _df, _cols in forecast_frames:
    if _df is None:
        for _c in _cols:
            base = base.withColumn(_c, F.lit(None).cast('bigint' if _c.endswith('_n_miembros') else 'double'))
    else:
        base = base.join(_df, 'fecha', 'left')

if not discharge_usable:
    print('caudal_m3s is not usable; publishing NULLs for caudal_actual_m3s')
    base = base.withColumn('caudal_actual_m3s', F.lit(None).cast('double'))

w = Window.orderBy('fecha')
w3 = w.rowsBetween(-2, 0)
w7 = w.rowsBetween(-6, 0)

# 8 horizontes (Decision 019: t+1..t+7, t+14), en paralelo para nivel y caudal -- 16
# columnas de target en total.
HORIZONS = [1, 2, 3, 4, 5, 6, 7, 14]

dataset = (
    base
    .withColumn('lluvia_agregado_alta_frontera_acum_3d_mm', F.sum('lluvia_acumulada_mm').over(w3))
    .withColumn('lluvia_agregado_alta_frontera_acum_7d_mm', F.sum('lluvia_acumulada_mm').over(w7))
    .withColumn('lluvia_merge_alta_frontera_acum_3d_mm', F.sum('lluvia_merge_alta_frontera_mm').over(w3))
    .withColumn('lluvia_merge_alta_frontera_acum_7d_mm', F.sum('lluvia_merge_alta_frontera_mm').over(w7))
    .withColumn('caudal_lag_1d', F.lag('caudal_actual_m3s', 1).over(w))
    .withColumn('caudal_lag_3d', F.lag('caudal_actual_m3s', 3).over(w))
    .withColumn('caudal_lag_7d', F.lag('caudal_actual_m3s', 7).over(w))
    .withColumn('caudal_media_3d', F.avg('caudal_actual_m3s').over(w3))
    .withColumn('caudal_media_7d', F.avg('caudal_actual_m3s').over(w7))
    .withColumn('caudal_delta_1d', F.col('caudal_actual_m3s') - F.col('caudal_lag_1d'))
    .withColumn('feature_generated_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
)

for h in HORIZONS:
    dataset = (
        dataset
        .withColumn(f'caudal_t_mas_{h}d', F.lead('caudal_actual_m3s', h).over(w))
    )

for subcuenca in SUBCUENCAS:
    col_m3s = f'caudal_agregado_{subcuenca}_m3s'
    dataset = (
        dataset
        .withColumn(f'caudal_agregado_{subcuenca}_lag_1d', F.lag(col_m3s, 1).over(w))
        .withColumn(f'caudal_agregado_{subcuenca}_lag_2d', F.lag(col_m3s, 2).over(w))
        .withColumn(f'caudal_agregado_{subcuenca}_lag_3d', F.lag(col_m3s, 3).over(w))
    )

output_columns = [
    'fecha', 'punto_prediccion', 'codigoestacao',
    'temp_media_c', 'temp_min_c', 'temp_max_c', 'temp_agregado_alta_frontera_station_count', 'temp_agregado_alta_frontera_cobertura_pct',
    'lluvia_acumulada_mm', 'lluvia_is_usable',
    'lluvia_agregado_alta_frontera_acum_3d_mm', 'lluvia_agregado_alta_frontera_acum_7d_mm',
    'lluvia_agregado_alta_frontera_station_count', 'lluvia_agregado_alta_frontera_cobertura_pct',
] + [
    'caudal_actual_m3s', 'caudal_registros_validos', 'caudal_metodo', 'caudal_extrapolado', 'distancia_fuera_rango_cm',
    'supera_aforo_maximo', 'caudal_confiable', 'curva_vigencia_extendida', 'caudal_lag_1d', 'caudal_lag_3d', 'caudal_lag_7d',
    'caudal_media_3d', 'caudal_media_7d', 'caudal_delta_1d',
] + [f'caudal_t_mas_{h}d' for h in HORIZONS]
for subcuenca in SUBCUENCAS:
    output_columns += [
        f'caudal_agregado_{subcuenca}_m3s', f'caudal_agregado_{subcuenca}_lag_1d',
        f'caudal_agregado_{subcuenca}_lag_2d', f'caudal_agregado_{subcuenca}_lag_3d',
        f'caudal_agregado_{subcuenca}_confiable_pct',
    ]
output_columns += [
    'lluvia_merge_alta_frontera_mm', 'lluvia_merge_alta_frontera_max_mm',
    'lluvia_merge_alta_frontera_acum_3d_mm', 'lluvia_merge_alta_frontera_acum_7d_mm',
    'lluvia_merge_alta_frontera_pluviometros', 'lluvia_merge_alta_frontera_cobertura_pct', 'lluvia_merge_alta_frontera_es_preliminar',
    'temp_samet_alta_frontera_media_c', 'temp_samet_alta_frontera_max_c', 'temp_samet_alta_frontera_min_c',
    'temp_samet_alta_frontera_cobertura_pct', 'temp_samet_alta_frontera_es_preliminar',
]
output_columns += forecast_columns
output_columns += ['feature_generated_at', 'updated_at']

dataset = dataset.select(*output_columns)

window_start, window_end = target_window(dataset)
print(f'Gold replacement window: {window_start} to {window_end}')
replace_target_window(dataset, window_start, window_end)

spark.table(TARGET_TABLE).filter(F.col('punto_prediccion') == F.lit(PUNTO_PREDICCION)).agg(F.min('fecha').alias('inicio'), F.max('fecha').alias('fin'), F.count('*').alias('rows')).show()